# CDN demo

This notebook trains CDN on the `breastw` dataset of ADBench with the protocol of the paper and reproduces its AUC-ROC and AUC-PR, which are listed together with the other 45 datasets in `results/cdn_adbench46.csv`.

In [1]:
import os, time, urllib.request
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from cdn import CDN

## Data

ADBench datasets are stored as `.npz` files with arrays `X` and `y` (1 = anomaly). The file is downloaded from the ADBench repository if it is not in `data/`.

In [2]:
NAME = '4_breastw'
URL = f'https://github.com/Minqi824/ADBench/raw/main/adbench/datasets/Classical/{NAME}.npz'
path = f'data/{NAME}.npz'
if not os.path.exists(path):
    os.makedirs('data', exist_ok=True)
    urllib.request.urlretrieve(URL, path)
data = np.load(path, allow_pickle=True)
X, y = data['X'], data['y']
print(f'{NAME}: n={X.shape[0]}, d={X.shape[1]}, anomalies={int(y.sum())}')

4_breastw: n=683, d=9, anomalies=239


## Protocol

Semi-supervised setting: the first half of the normal samples is the training set; the test set contains the remaining normal samples and all anomalies. Features are standardised with statistics of the training set.

In [3]:
X_normal, X_anom = X[y == 0], X[y == 1]
n_train = len(X_normal) // 2
X_train = X_normal[:n_train]
X_test = np.vstack([X_normal[n_train:], X_anom])
y_test = np.r_[np.zeros(len(X_normal) - n_train), np.ones(len(X_anom))]

scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
print('train', X_train.shape, 'test', X_test.shape)

train (222, 9) test (461, 9)


## Train and evaluate CDN

The default parameters are the single configuration used for all datasets in the paper. Results are averaged over five seeds.

In [4]:
SEEDS = [42, 123, 456, 789, 1024]
roc, pr = [], []
for seed in SEEDS:
    model = CDN(random_state=seed).fit(X_train)
    scores = model.decision_function(X_test)
    roc.append(roc_auc_score(y_test, scores))
    pr.append(average_precision_score(y_test, scores))
print(f'AUC-ROC {np.mean(roc):.4f} ± {np.std(roc):.4f}')
print(f'AUC-PR  {np.mean(pr):.4f} ± {np.std(pr):.4f}')

AUC-ROC 0.9924 ± 0.0001
AUC-PR  0.9919 ± 0.0001


In [5]:
ref = pd.read_csv('results/cdn_adbench46.csv', index_col='dataset').loc['breastw']
print('reported in the paper:')
print(f"AUC-ROC {ref.auc_roc_mean:.4f} ± {ref.auc_roc_std:.4f}")
print(f"AUC-PR  {ref.auc_pr_mean:.4f} ± {ref.auc_pr_std:.4f}")

reported in the paper:
AUC-ROC 0.9924 ± 0.0001
AUC-PR  0.9919 ± 0.0001


## Inference

Scoring needs only the trained network: one forward pass per query, no neighbour search and no access to the training data.

In [6]:
n_params = sum(p.numel() for p in model.net_.parameters())
model.decision_function(X_test)  # warm-up
t0 = time.perf_counter()
model.decision_function(X_test)
t = time.perf_counter() - t0
print(f'{n_params} parameters, {len(X_test)} queries scored in {1000 * t:.2f} ms')

2825 parameters, 461 queries scored in 0.78 ms
